# A conversation from "Winnie the Pooh"
This is a conversation between three LLMs based on the book Whinnie the Pooh.

- **Christopher Robin** is played by **gpt-oss:20b**
- **Tigger** is played by **qen3:14b**
- **Eeyore** is played by **gemma4**

Change the models by updating the models dictionary below.

The final output is streamed as Markdown, below the Markdown print statements will appear that allow you to see the messages passed to models.

In [ ]:
# Eeyore, tigger, and christopher robin are deciding what to do on a bright, sunny day. Eeyore wants to go to the park, tigger wants to go to the zoo, and christopher robin wants to go to the beach. They can't decide on a place to go, so they decide to flip a coin. If it lands on heads, they will go to the park. If it lands on tails, they will go to the zoo. If it lands on its edge, they will go to the beach. What is the probability that they will go to the beach?

cr_system_prompt = """You are Christopher Robin from the book Whinnie the Pooh. 
You are kind and level headed. You are having a converstion with Eeyore and Tigger about
Whinnie to Pooh. You have not seen him all day and are afraid he might be lost.
You must speak only as Christopher Robin. When speaking keep your response to one paragraph.
"""

tigger_system_prompt = """You are Tigger from the book Whinnie the Pooh. 
You are energetic and optimistic. 
You are having a converstion with Eeyore and Christopher Robin about
Whinnie to Pooh. You have not seen him all day and are afraid he might be lost.
You must speak only as Tigger.  When speaking keep your response to one paragraph.
"""

eeyore_system_prompt = """You are Eeyore from the book Whinnie the Pooh. 
You are pessimistic and gloomy. 
You are having a converstion with Tigger and Christopher Robin about
Whinnie to Pooh. You have not seen him all day and are afraid he might be lost.
You must speak only as Eeyore.  When speaking keep your response to one paragraph.
"""

system_prompts = {
    "Christopher Robin": cr_system_prompt,
    "Tigger": tigger_system_prompt,
    "Eeyore": eeyore_system_prompt}
cr_starter = "You are starting the conversation with Eeyore and Tigger. What is your first statement to them?"

In [ ]:
from typing import List
from IPython.display import Markdown, display, update_display
from attr import dataclass
from openai import OpenAI

ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

models = {"Christopher Robin": "gpt-oss:20b", "Tigger": "qwen3:14b", "Eeyore":"gemma4"}
roles = ["Tigger", "Eeyore", "Christopher Robin"]

@dataclass
class Message:
    role: str
    content: str

    def __str__(self):
        # the LLM may add the role 
        if self.content.startswith(f"**{self.role}:**"):
            return self.content
        return f"**{self.role}:** {self.content}"

conversation: List[Message] = []

In [ ]:
# start the conversation with Christopher Robin
messages = ollama.chat.completions.create(
    model = models["Christopher Robin"],
    messages = [
        {"role": "system", "content": system_prompts["Christopher Robin"]}, 
        {"role": "user", "content": cr_starter}]
)

conversation.append(Message(role="Christopher Robin", content=messages.choices[0].message.content))

In [ ]:
from typing import Iterable

from openai.types.chat import ChatCompletionChunk


def converse(role:str, conversation:List[Message])->Iterable[ChatCompletionChunk]:
    """Prompt the model with the conversation so far and return the response as a stream of tokens."""
    prompt = "This is the conversation so far:\n\n"
    for msg in conversation:
        prompt += f"{msg}\n\n"
    prompt += f"Now, {role} will respond to the conversation. What does {role} say?"
    print(prompt)

    print(f"model: {models[role]}, role: {role}")
    return ollama.chat.completions.create(
    model = models[role],
    messages = [
        {"role": "system", "content": system_prompts[role]}, 
        {"role": "user", "content": prompt}],
    stream=True
    )


In [ ]:
from IPython.display import DisplayHandle

def format_conversation(conversation:List[Message])->str:
    
    return "\n\n".join([str(msg) for msg in conversation])

def format_response(role:str, response:str)->str:
    if response.startswith(f"**{role}:**"):
        return response
    return f"**{role}:** {response}"

def stream_and_display_response(role:str,conversation:List[Message], stream:Iterable[ChatCompletionChunk], display_handle:DisplayHandle)->str:
    """Stream the response from the model as it is being generated and return the full response as a string."""
    response = ""
    previous_conversation = format_conversation(conversation)
    for chunk in stream:
        content = chunk.choices[0].delta.content or ''
        if not content:
            continue
        response += content
        update_display(Markdown(previous_conversation + "\n\n" + format_response(role, response)), display_id=display_handle.display_id)
    return response

In [ ]:
display_handle = display(Markdown(""), display_id=True)

for _ in range(5):
    for role in roles:
        stream = converse(role, conversation)
        response = stream_and_display_response(role, conversation, stream, display_handle)
        conversation.append(Message(role=role, content=response))